# 03 · Visualize sales and margin
Three complementary views: revenue and gross profit over time, category revenue,
and monthly gross margin. The saved figure and executed notebook belong to this run.

In [ ]:
# The runner injects these values. For manual use, copy run_id from notebook 01.
run_id = None
input_csv = None
demo_failure = None

In [ ]:
from notebook_pipeline import start_notebook

session = start_notebook("03_visualize", run_id=run_id)
run_id = session.run_id
import json

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter, PercentFormatter

In [ ]:
# step: load_analysis
monthly = pd.read_parquet(session.input("02_analyze", "monthly_sales.parquet"))
categories = pd.read_parquet(session.input("02_analyze", "category_sales.parquet"))
kpis = json.loads(session.input("02_analyze", "kpis.json").read_text(encoding="utf-8"))

In [ ]:
# step: configure_figure
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
    }
)
# Disable automatic display until all panels are ready.
plt.ioff()
fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
fig.patch.set_facecolor("#f8fafc")
fig.suptitle("Retail performance | January–June 2025", fontsize=21, fontweight="bold")
money = FuncFormatter(lambda value, _: f"${value / 1000:,.0f}k")

In [ ]:
# step: plot_monthly_revenue
labels = monthly["month"].dt.strftime("%b")
axes[0].plot(
    labels, monthly["revenue"], color="#2563eb", marker="o", linewidth=2.5, label="Revenue"
)
axes[0].plot(
    labels,
    monthly["gross_profit"],
    color="#0d9488",
    marker="o",
    linewidth=2.5,
    label="Gross profit",
)
axes[0].set(title="Growth over time", ylabel="USD", ylim=(0, None))
axes[0].yaxis.set_major_formatter(money)
axes[0].legend(frameon=False)

In [ ]:
# step: plot_category_revenue
ordered = categories.sort_values("revenue")
axes[1].barh(ordered["category"], ordered["revenue"], color="#2563eb", height=0.55)
axes[1].set(title="Revenue by category", xlabel="USD")
axes[1].xaxis.set_major_formatter(money)

In [ ]:
# step: plot_gross_margin
axes[2].plot(labels, monthly["gross_margin"], color="#7c3aed", marker="o", linewidth=2.5)
axes[2].axhline(kpis["gross_margin"], linestyle="--", color="#64748b", label="Period margin")
axes[2].yaxis.set_major_formatter(PercentFormatter(1))
axes[2].set(title="Gross margin", ylim=(0, max(0.5, float(monthly["gross_margin"].max()) + 0.05)))
axes[2].legend(frameon=False)
for axis in axes:
    axis.grid(axis="y", alpha=0.15)
fig

In [ ]:
# step: save_dashboard
with session.output("sales_dashboard.png") as path:
    fig.savefig(path, dpi=160, facecolor=fig.get_facecolor())
plt.close(fig)
print(f"Top category: {kpis['top_category']}; period gross margin: {kpis['gross_margin']:.1%}")

In [ ]:
# step: finish_visualization
print(f"Dashboard: {session.run.path / 'artifacts/03_visualize/sales_dashboard.png'}")
session.finish()